# 12 (Mac) - ten hospitals: organism_community (n=10), local run

Same protocol as `12_ten_hospitals.ipynb` (six methods, five seeds, identical budgets), run on a Mac
CPU instead of Colab. Setup assumes the repo is already checked out on `feat/federated-baselines` and
the venv from `README.md` is active. Results go to `results/amr_org10_full.json` (not in `.gitignore`, so don't `git add` it). Resumes from that file if it exists.

Needs `ARMD_DIR` pointing at a local ARMD folder with the 10 CSVs.


In [1]:
# 1) Check the repo branch + deps (run once in a terminal, not here, if already installed)
# cd Capstone-amr-fed && git checkout feat/federated-baselines && git pull
# source .venv/bin/activate && pip install -r requirements.txt && pip install -e . 'flwr[simulation]' torch_geometric
import sys, torch, ray, flwr, torch_geometric
print('python', sys.version.split()[0], '| torch', torch.__version__, '| flwr', flwr.__version__,
      '| ray', ray.__version__, '| torch_geometric', torch_geometric.__version__)
print('device:', 'cuda' if torch.cuda.is_available() else 'cpu')


/Users/vikramadityasharma/Desktop/Projects/Capstone-amr-fed/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-10-25 14:18:55,076	INFO util.py:155 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
/Users/vikramadityasharma/Desktop/Projects/Capstone-amr-fed/.venv/lib/python3.13/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


python 3.13.4 | torch 2.14.1 | flwr 1.42.0 | ray 2.59.0 | torch_geometric 2.8.1
device: cpu


In [2]:
# 2) Point at the local ARMD folder (set ARMD_DIR in your shell or edit here)
import os
from pathlib import Path
os.environ.setdefault('ARMD_DIR', '/Users/vikramadityasharma/Library/CloudStorage/GoogleDrive-vikramaditya.raunak@gmail.com/.shortcut-targets-by-id/1uVx5PmldPyJBMtnB9Lkl9N1rNBQrmJt-/ARMD')  # edit if your folder differs
print('ARMD_DIR:', os.environ['ARMD_DIR'])


ARMD_DIR: /Users/vikramadityasharma/Library/CloudStorage/GoogleDrive-vikramaditya.raunak@gmail.com/.shortcut-targets-by-id/1uVx5PmldPyJBMtnB9Lkl9N1rNBQrmJt-/ARMD


In [3]:
# 3) Sanity: data resolves + imports
import sys
from pathlib import Path
# find the repo's src/ folder whether the notebook runs from notebooks/ or the repo root
SRC = next(p / 'src' for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src' / 'amr_fed').is_dir())
sys.path.insert(0, str(SRC))
from amr_fed import config
D = Path(config.DATA_DIR)
print('DATA_DIR:', D, '| exists:', D.exists())
assert D.exists(), 'ARMD_DIR is wrong — fix cell 2 and re-run.'
from amr_fed.federated.run import run_multiseed
from amr_fed.data_loader import load_cohort_frame, PK
from amr_fed.partition import louvain_split, partition_summary
print('imports OK')

ModuleNotFoundError: No module named 'amr_fed'

In [ ]:
# 4) Load cohort ONCE — reused by every arm cell below
import sys as _s
for _m in [m for m in list(_s.modules) if m.startswith('amr_fed')]:
    del _s.modules[_m]
from amr_fed.data_loader import load_cohort_frame
df = load_cohort_frame()
print('rows:', f'{len(df):,}')

In [ ]:
# 5) SMOKE: organism_community with 10 hospitals (deterministic -> same partition for every seed)
import sys as _s
for _m in [m for m in list(_s.modules) if m.startswith('amr_fed')]:
    del _s.modules[_m]
from amr_fed.data_loader import PK
from amr_fed.partition import organism_community, partition_summary
SEEDS = (42, 43, 44, 45, 46)
assign = organism_community(df, n_clients=10)
g, niid = partition_summary(df, assign)
print(g)
print(f'actual hospitals: {assign.nunique()} | non-IID(wstd): {niid:.4f}')
assert assign.nunique() == 10, 'did not produce 10 hospitals - stop and report'
assert assign.index.is_unique and len(assign) == df[PK].nunique(), 'not 1:1'
assert (g['n_tests'] / g['n_tests'].sum() >= 0.01).all(), 'hospital below 1% size floor'
print('SMOKE PASS')


In [ ]:
# 6) All arms, identical protocol (rounds=8, local_epochs=4, local_only_epochs=32, hidden=128, 5 seeds).
#    Pooled is computed once (first arm); later arms skip it to save time.
import os, ray, json
os.environ.pop('AMR_RNG_PROBE', None)
import sys as _s
for _m in [m for m in list(_s.modules) if m.startswith('amr_fed')]:
    del _s.modules[_m]
from amr_fed.federated.run import run_multiseed
from amr_fed.partition import organism_community

KW = dict(seeds=SEEDS, df=df, partition_fn=lambda d, s: organism_community(d, n_clients=10, seed=s),
          n_clients=10, rounds=8, local_epochs=4, local_only_epochs=32, hidden=128)
ARMS = [  # (key, label, extra kwargs)
    ('fedavg',        'org10/FedAvg',        dict(strategy='fedavg')),
    ('fedprox',       'org10/FedProx',       dict(strategy='fedprox')),
    ('fedmedian',     'org10/FedMedian',     dict(strategy='fedmedian')),
    ('fedtrimmedavg', 'org10/FedTrimmedAvg', dict(strategy='fedtrimmedavg')),
    ('uniform',       'org10/UNIFORM (T=inf control)', dict(strategy='topology', topo_mode='distinctiveness', topo_temperature=float('inf'))),
    ('topo',          'org10/TOPO (T=1)',    dict(strategy='topology', topo_mode='distinctiveness', topo_temperature=1.0)),
]
RESULTS = SRC.parent / 'results' / 'amr_org10_full.json'  # repo root, whatever the notebook's cwd
RESULTS.parent.mkdir(exist_ok=True)
OUT = str(RESULTS)
# Resume: arms already checkpointed in OUT are loaded, not rerun.
res = json.load(open(OUT)) if os.path.exists(OUT) else {}
for i, (key, label, extra) in enumerate(ARMS):
    if key in res:
        print(f'skip {key}: already saved')
        continue
    ray.shutdown()
    res[key] = run_multiseed(label=label, compute_pooled=(i == 0), **KW, **extra)
    json.dump(res, open(OUT, 'w'), default=str)  # checkpoint after every arm
    print(f'checkpoint saved after {key}')


In [ ]:
# 7) Comparison. Partition is fixed -> hospital ids line up across seeds. macro-F1, best round.
import numpy as np
keys = [k for k, _, _ in ARMS]
names = {'fedavg':'FedAvg','fedprox':'FedProx','fedmedian':'FedMed','fedtrimmedavg':'FedTrim','uniform':'Uniform','topo':'Topo'}
runs = {k: res[k]['runs'] for k in keys}
n = len(runs['fedavg'][0]['sizes'])
def mean(k, key, h): return float(np.mean([r[key][h] for r in runs[k]]))

print('=== per hospital: n | local pooled | each method | (method - local) ===')
print(f"{'H':<3}{'n_pat':>7} | {'local':>6} {'pooled':>6} | " + ' '.join(f'{names[k]:>7}' for k in keys) + ' | ' + ' '.join(f'{names[k][:6]+"-L":>8}' for k in keys))
for h in range(n):
    lo = mean('fedavg','local_only_per_client',h); po = mean('fedavg','pooled_per_client',h)
    ms = [mean(k,'fedavg_per_client',h) for k in keys]
    print(f"H{h:<2}{runs['fedavg'][0]['sizes'][h]:>7} | {lo:>6.4f} {po:>6.4f} | " + ' '.join(f'{m:>7.4f}' for m in ms) + ' | ' + ' '.join(f'{m-lo:>+8.4f}' for m in ms))

print('\n=== summary (mean +/- std over seeds) ===')
def f(ms): return 'n/a' if ms is None or ms[0] is None else f'{ms[0]:.4f} +/- {ms[1]:.4f}'
print(f"{'method':<14} {'best-round':<18} {'final-round':<18} {'worst-hospital':<18} {'AUROC':<18}")
print(f"{'local-only':<14} {f(res['fedavg']['local_only']):<18} {'-':<18} {f(res['fedavg']['worst_local']):<18} {f(res['fedavg']['local_only_auroc']):<18}")
print(f"{'pooled':<14} {f(res['fedavg']['pooled']):<18} {'-':<18} {f(res['fedavg']['pooled_worst']):<18} {f(res['fedavg']['pooled_auroc']):<18}")
for k in keys:
    r = res[k]
    print(f"{names[k]:<14} {f(r['fedavg_best']):<18} {f(r['fedavg_final']):<18} {f(r['worst_fed']):<18} {f(r['fedavg_auroc_best']):<18}")

print('\n=== hospitals where each method beats local-only (mean over seeds) ===')
for k in keys:
    wins = sum(mean(k,'fedavg_per_client',h) > mean('fedavg','local_only_per_client',h) for h in range(n))
    print(f'{names[k]:<9} {wins}/{n}')
